# Consideraciones de Producción para Sistemas RAG

En los notebooks anteriores construimos los componentes técnicos del pipeline: chunking, retrieval y reranking. Ahora viene la parte que casi ningún tutorial menciona: **¿qué pasa cuando lo pones en producción?**

Este notebook está pensado para principiantes que ya entienden cómo funciona RAG y quieren saber cómo mantenerlo funcionando bien en el tiempo.

**Lo que aprenderás:**
1. Cómo añadir logging para saber qué está pasando
2. Cómo medir la latencia de cada etapa
3. Cómo manejar errores sin que el sistema se caiga
4. Cómo actualizar los documentos del índice sin reiniciarlo todo
5. Cómo guardar y versionar configuraciones
6. Los 4 fallos más comunes y cómo detectarlos

**Solo usaremos la librería estándar de Python** — sin dependencias extra.

## ¿Qué significa "producción"?

Un sistema RAG en producción es diferente a uno en un notebook de pruebas:

| Entorno | Características |
|---|---|
| Notebook de pruebas | Un usuario, datos fijos, puedes reiniciar cuando quieras |
| Producción | Muchos usuarios simultáneos, datos que cambian, no puedes parar el servicio |

Los tres problemas principales que aparecen en producción:

- **Visibilidad**: sin logs no sabes por qué algo falla
- **Resiliencia**: el sistema tiene que funcionar aunque un componente falle
- **Mantenimiento**: los documentos cambian, los modelos se actualizan, la configuración evoluciona

In [1]:
# Solo librería estándar — no hace falta instalar nada
import json
import logging
import os
import time
import hashlib
import random
from contextlib import contextmanager
from dataclasses import dataclass, asdict, field
from datetime import datetime
from pathlib import Path
from typing import Dict, List, Optional, Any

print('Todo listo, solo librería estándar de Python.')

Todo listo, solo librería estándar de Python.


## 1. Logging: Saber Qué Está Pasando

El error más común al pasar a producción es no tener logs. Cuando algo falla en producción, los logs son la única forma de saber qué pasó.

**Regla básica**: si algo tarda más de lo esperado o falla, tiene que quedar registrado.

Vamos a configurar logging con un formato útil para producción: **timestamp + nivel + componente + mensaje**.

In [2]:
def configurar_logging(nivel: str = 'INFO') -> logging.Logger:
    """Configura logging con formato legible y útil para producción."""
    formato = '%(asctime)s | %(levelname)-8s | %(name)-20s | %(message)s'
    fecha_formato = '%Y-%m-%d %H:%M:%S'

    logging.basicConfig(
        level=getattr(logging, nivel.upper()),
        format=formato,
        datefmt=fecha_formato,
        handlers=[logging.StreamHandler()],  # en producción añadir FileHandler
    )
    return logging.getLogger('rag')


logger = configurar_logging('INFO')

# ── Simulamos el ciclo de vida de una consulta ─────────────────────────────────
query_id = 'q-001'
query    = '¿Qué es el reranking en sistemas RAG?'

logger.info(f'[{query_id}] Nueva consulta recibida: "{query[:60]}"')
logger.info(f'[{query_id}] Retrieval completado: 20 candidatos en 12.3ms')
logger.info(f'[{query_id}] Reranking completado: top-5 en 87.4ms')
logger.info(f'[{query_id}] Respuesta generada: 312 tokens en 1.2s')
logger.warning(f'[{query_id}] Latencia alta: total 1.4s (umbral: 1.0s)')
logger.error(f'[{query_id}] Fallo simulado: timeout al conectar con la API del LLM')

2026-05-18 16:57:31 | INFO     | rag                  | [q-001] Nueva consulta recibida: "¿Qué es el reranking en sistemas RAG?"
2026-05-18 16:57:31 | INFO     | rag                  | [q-001] Retrieval completado: 20 candidatos en 12.3ms
2026-05-18 16:57:31 | INFO     | rag                  | [q-001] Reranking completado: top-5 en 87.4ms
2026-05-18 16:57:31 | INFO     | rag                  | [q-001] Respuesta generada: 312 tokens en 1.2s
2026-05-18 16:57:31 | WARNING  | rag                  | [q-001] Latencia alta: total 1.4s (umbral: 1.0s)
2026-05-18 16:57:31 | ERROR    | rag                  | [q-001] Fallo simulado: timeout al conectar con la API del LLM


### 1.1 Midiendo Latencia por Etapa

No basta con saber que una consulta fue lenta — necesitas saber **qué etapa** fue lenta. ¿Fue el retrieval? ¿El reranking? ¿El LLM?

Vamos a crear un medidor de tiempo reutilizable con un **context manager** (el patrón `with`). Un context manager ejecuta código automáticamente al entrar y al salir del bloque, lo que lo hace perfecto para medir tiempos.

In [3]:
class Cronometro:
    """Mide el tiempo de cada etapa del pipeline y guarda el historial."""

    def __init__(self):
        self.etapas: Dict[str, List[float]] = {}  # nombre -> [tiempos en ms]

    @contextmanager
    def medir(self, etapa: str):
        """Context manager: mide el tiempo del bloque `with`."""
        inicio = time.perf_counter()
        try:
            yield
        finally:
            tiempo_ms = (time.perf_counter() - inicio) * 1000
            self.etapas.setdefault(etapa, []).append(tiempo_ms)
            logger.info(f'  {etapa}: {tiempo_ms:.1f}ms')

    def resumen(self) -> None:
        """Imprime un resumen de latencias acumuladas."""
        print(f'\n{"Etapa":<22} {"Llamadas":>9} {"Media (ms)":>11} {"Máx (ms)":>10}')
        print('-' * 57)
        for etapa, tiempos in self.etapas.items():
            media = sum(tiempos) / len(tiempos)
            maximo = max(tiempos)
            print(f'{etapa:<22} {len(tiempos):>9} {media:>11.1f} {maximo:>10.1f}')


# ── Demo: simulamos 3 consultas y vemos la latencia por etapa ─────────────────
cronometro = Cronometro()
random.seed(42)

for i in range(3):
    logger.info(f'--- Consulta {i+1} ---')

    with cronometro.medir('bm25'):
        time.sleep(random.uniform(0.001, 0.003))  # simula BM25

    with cronometro.medir('dense_faiss'):
        time.sleep(random.uniform(0.004, 0.008))  # simula búsqueda densa

    with cronometro.medir('cross_encoder'):
        time.sleep(random.uniform(0.05, 0.12))    # simula cross-encoder

    with cronometro.medir('llm_generacion'):
        time.sleep(random.uniform(0.8, 1.5))      # simula llamada al LLM

cronometro.resumen()

2026-05-18 16:57:31 | INFO     | rag                  | --- Consulta 1 ---
2026-05-18 16:57:31 | INFO     | rag                  |   bm25: 2.5ms
2026-05-18 16:57:31 | INFO     | rag                  |   dense_faiss: 4.3ms
2026-05-18 16:57:31 | INFO     | rag                  |   cross_encoder: 69.6ms
2026-05-18 16:57:32 | INFO     | rag                  |   llm_generacion: 960.9ms
2026-05-18 16:57:32 | INFO     | rag                  | --- Consulta 2 ---
2026-05-18 16:57:32 | INFO     | rag                  |   bm25: 2.6ms
2026-05-18 16:57:32 | INFO     | rag                  |   dense_faiss: 6.8ms
2026-05-18 16:57:32 | INFO     | rag                  |   cross_encoder: 113.1ms
2026-05-18 16:57:33 | INFO     | rag                  |   llm_generacion: 864.2ms
2026-05-18 16:57:33 | INFO     | rag                  | --- Consulta 3 ---
2026-05-18 16:57:33 | INFO     | rag                  |   bm25: 3.2ms
2026-05-18 16:57:33 | INFO     | rag                  |   dense_faiss: 4.2ms
2026-05-1


Etapa                   Llamadas  Media (ms)   Máx (ms)
---------------------------------------------------------
bm25                           3         2.7        3.2
dense_faiss                    3         5.1        6.8
cross_encoder                  3        82.7      113.1
llm_generacion                 3       994.7     1159.0


## 2. Manejo de Errores y Fallbacks

En producción, los componentes fallan. El modelo de embedding puede tardar demasiado, la API del LLM puede estar caída, FAISS puede quedarse sin memoria.

El principio clave es: **degradación elegante**. Si el reranker falla, devuelve los resultados sin rerankear. Si el LLM falla, muestra un mensaje claro. El sistema nunca debe caerse completamente por un fallo en un componente secundario.

Hay tres patrones básicos:
- **Try/except con fallback**: si el método principal falla, usa uno más simple
- **Timeout**: si algo tarda demasiado, cancélalo y sigue
- **Retry con backoff**: reintenta con espera creciente antes de rendirse

In [4]:
import signal


class TimeoutError(Exception):
    pass


@contextmanager
def timeout(segundos: int):
    """Context manager que lanza TimeoutError si el bloque tarda más de `segundos`."""
    def _handler(signum, frame):
        raise TimeoutError(f'Operación cancelada tras {segundos}s')

    signal.signal(signal.SIGALRM, _handler)
    signal.alarm(segundos)
    try:
        yield
    finally:
        signal.alarm(0)  # cancelar la alarma si terminamos antes


def retry(fn, intentos: int = 3, espera_base: float = 0.5):
    """Reintenta `fn` hasta `intentos` veces con espera exponencial."""
    for intento in range(intentos):
        try:
            return fn()
        except Exception as e:
            if intento == intentos - 1:
                raise  # último intento: propagar el error
            espera = espera_base * (2 ** intento)  # 0.5s, 1s, 2s...
            logger.warning(f'Intento {intento+1} fallido: {e}. Reintentando en {espera:.1f}s...')
            time.sleep(espera)


# ── Demo: pipeline con degradación elegante ───────────────────────────────────
def reranker_principal(candidatos: List[str]) -> List[str]:
    """Simula un reranker que a veces falla."""
    if random.random() < 0.6:  # falla el 60% de las veces (para el demo)
        raise RuntimeError('Cross-encoder: modelo no disponible')
    return candidatos  # en un caso real, devolvería reordenados


def reranker_fallback(candidatos: List[str]) -> List[str]:
    """Fallback: devuelve los candidatos tal cual, sin rerankear."""
    logger.warning('Usando fallback: devolviendo candidatos sin rerankear')
    return candidatos


def recuperar_con_fallback(query: str) -> List[str]:
    """Pipeline de recuperación que no se cae aunque el reranker falle."""
    candidatos = ['doc_1', 'doc_2', 'doc_3', 'doc_4', 'doc_5']  # simulado

    try:
        resultados = reranker_principal(candidatos)
        logger.info('Reranker principal usado correctamente')
        return resultados
    except Exception as e:
        logger.error(f'Reranker principal falló: {e}')
        return reranker_fallback(candidatos)


random.seed(0)
print('Ejecutando 4 consultas con el reranker inestable:\n')
for i in range(4):
    resultado = recuperar_con_fallback(f'consulta_{i}')
    print(f'  Consulta {i+1}: {len(resultado)} documentos devueltos\n')

2026-05-18 16:57:34 | INFO     | rag                  | Reranker principal usado correctamente
2026-05-18 16:57:34 | INFO     | rag                  | Reranker principal usado correctamente
2026-05-18 16:57:34 | ERROR    | rag                  | Reranker principal falló: Cross-encoder: modelo no disponible
2026-05-18 16:57:34 | WARNING  | rag                  | Usando fallback: devolviendo candidatos sin rerankear
2026-05-18 16:57:34 | ERROR    | rag                  | Reranker principal falló: Cross-encoder: modelo no disponible
2026-05-18 16:57:34 | WARNING  | rag                  | Usando fallback: devolviendo candidatos sin rerankear


Ejecutando 4 consultas con el reranker inestable:

  Consulta 1: 5 documentos devueltos

  Consulta 2: 5 documentos devueltos

  Consulta 3: 5 documentos devueltos

  Consulta 4: 5 documentos devueltos



## 3. Actualización de Índices

En producción los documentos cambian constantemente: se publican nuevos artículos, se actualizan páginas, se eliminan documentos obsoletos.

**El problema**: si tienes un corpus de 100.000 documentos, no puedes reindexar todo cada vez que cambia uno.

**La solución**: detección de cambios con hashes. Antes de reindexar un documento, comprobamos si su contenido cambió realmente. Si no cambió, nos ahorramos el trabajo.

Un **hash** es una huella digital del contenido: si el texto cambia aunque sea una letra, el hash es completamente diferente.

In [5]:
# ── Primero, veamos cómo funciona un hash ─────────────────────────────────────
texto_original = 'El Quijote es una novela española del siglo XVII.'
texto_modificado = 'El Quijote es una novela española del siglo XVII!'

def calcular_hash(texto: str) -> str:
    return hashlib.sha256(texto.encode('utf-8')).hexdigest()[:16]  # primeros 16 chars

print('Hash del original: ', calcular_hash(texto_original))
print('Hash del modificado:', calcular_hash(texto_modificado))
print('(cambia completamente aunque solo cambie el punto final)')

Hash del original:  77dba35c70c92a97
Hash del modificado: 746df80b931ddcf8
(cambia completamente aunque solo cambie el punto final)


In [6]:
@dataclass
class Documento:
    id: str
    contenido: str
    metadatos: Dict[str, Any] = field(default_factory=dict)
    hash: str = ''
    actualizado_en: float = field(default_factory=time.time)


class GestorDocumentos:
    """
    Gestiona un conjunto de documentos con detección de cambios.

    En un sistema real, aquí también actualizarías el índice FAISS
    y el índice BM25. Para el tutorial, nos centramos en la lógica
    de detección de cambios.
    """

    def __init__(self):
        self.documentos: Dict[str, Documento] = {}
        self._cambios_pendientes: List[str] = []  # IDs que necesitan reindexar

    @staticmethod
    def _hash(contenido: str) -> str:
        return hashlib.sha256(contenido.encode('utf-8')).hexdigest()

    def añadir_o_actualizar(self, doc_id: str, contenido: str,
                             metadatos: Optional[Dict] = None) -> str:
        """
        Añade o actualiza un documento. Devuelve 'añadido', 'actualizado'
        o 'sin_cambios'.
        """
        nuevo_hash = self._hash(contenido)

        if doc_id in self.documentos:
            if self.documentos[doc_id].hash == nuevo_hash:
                return 'sin_cambios'  # el contenido no cambió, no hacer nada
            estado = 'actualizado'
        else:
            estado = 'añadido'

        self.documentos[doc_id] = Documento(
            id=doc_id,
            contenido=contenido,
            metadatos=metadatos or {},
            hash=nuevo_hash,
        )
        self._cambios_pendientes.append(doc_id)
        return estado

    def eliminar(self, doc_id: str) -> bool:
        if doc_id not in self.documentos:
            return False
        del self.documentos[doc_id]
        logger.info(f'Documento eliminado: {doc_id}')
        return True

    def documentos_modificados_desde(self, timestamp: float) -> List[Documento]:
        """Devuelve los documentos actualizados después de `timestamp`."""
        return [
            doc for doc in self.documentos.values()
            if doc.actualizado_en >= timestamp
        ]

    def estadisticas(self) -> None:
        print(f'Documentos en el gestor: {len(self.documentos)}')
        print(f'Pendientes de reindexar: {len(self._cambios_pendientes)}')


# ── Demo ──────────────────────────────────────────────────────────────────────
gestor = GestorDocumentos()

operaciones = [
    ('doc-001', 'Don Quijote es el protagonista de la novela de Cervantes.'),
    ('doc-002', 'Sancho Panza es el escudero fiel de Don Quijote.'),
    ('doc-001', 'Don Quijote es el protagonista de la novela de Cervantes.'),  # sin cambios
    ('doc-001', 'Don Quijote de la Mancha es el protagonista de la novela de Cervantes.'),  # modificado
    ('doc-003', 'Dulcinea del Toboso es el amor idealizado de Don Quijote.'),
]

print('Procesando documentos:\n')
for doc_id, contenido in operaciones:
    estado = gestor.añadir_o_actualizar(doc_id, contenido)
    icono = {'añadido': '✚', 'actualizado': '↺', 'sin_cambios': '─'}[estado]
    print(f'  {icono} {doc_id}: {estado} — "{contenido[:50]}..."')

print()
gestor.estadisticas()

Procesando documentos:

  ✚ doc-001: añadido — "Don Quijote es el protagonista de la novela de Cer..."
  ✚ doc-002: añadido — "Sancho Panza es el escudero fiel de Don Quijote...."
  ─ doc-001: sin_cambios — "Don Quijote es el protagonista de la novela de Cer..."
  ↺ doc-001: actualizado — "Don Quijote de la Mancha es el protagonista de la ..."
  ✚ doc-003: añadido — "Dulcinea del Toboso es el amor idealizado de Don Q..."

Documentos en el gestor: 3
Pendientes de reindexar: 4


### 3.1 Tres Estrategias de Actualización

Dependiendo del tamaño de tu corpus y la frecuencia de cambios, hay tres enfoques:

| Estrategia | Cuándo usarla | Ventaja | Desventaja |
|---|---|---|---|
| **Reindexación completa** | Corpus pequeño (<10K docs) o cambios masivos | Simple, consistente | Sin servicio durante la reindexación |
| **Actualización incremental** | Cambios frecuentes y pequeños | Sin downtime | El índice puede quedar fragmentado |
| **Blue-Green (dos índices)** | Sistemas críticos sin downtime | Rollback instantáneo | Doble uso de memoria |

**Para empezar**: reindexación completa programada (p.ej. cada noche). Es lo más simple y correcto para la mayoría de casos.

**Cuándo pasar a incremental**: cuando el corpus supere los 100K documentos o los usuarios noten que la información está desactualizada durante horas.

## 4. Configuración y Versionado

Al principio, los parámetros del sistema (tamaño del chunk, modelo de embedding, top-K...) los ponemos directamente en el código. Esto es un error en producción: si cambias un parámetro tienes que modificar el código, hacer un deploy, y si algo sale mal, hacer rollback al deploy anterior.

La solución es **externalizar la configuración**: guardarla en un archivo JSON separado. Así puedes:
- Cambiar parámetros sin tocar el código
- Guardar el historial de qué configuración usabas en cada fecha
- Volver a una configuración anterior si algo va mal

In [7]:
@dataclass
class ConfigChunking:
    estrategia: str = 'semantico'   # 'fijo', 'parrafo', 'semantico'
    tamano_chunk: int = 512
    overlap: int = 50


@dataclass
class ConfigRetrieval:
    modelo_embedding: str = 'all-MiniLM-L6-v2'
    top_k_retrieval: int = 20
    top_k_final: int = 5
    usar_reranking: bool = True


@dataclass
class ConfigRAG:
    version: str
    descripcion: str
    chunking: ConfigChunking = field(default_factory=ConfigChunking)
    retrieval: ConfigRetrieval = field(default_factory=ConfigRetrieval)


class GestorConfiguracion:
    """Guarda y carga versiones de configuración en disco."""

    def __init__(self, directorio: str = '/tmp/rag_configs'):
        self.directorio = Path(directorio)
        self.directorio.mkdir(parents=True, exist_ok=True)

    def guardar(self, config: ConfigRAG) -> Path:
        ruta = self.directorio / f'config_v{config.version}.json'
        with open(ruta, 'w') as f:
            json.dump(asdict(config), f, indent=2, ensure_ascii=False)
        logger.info(f'Configuración v{config.version} guardada en {ruta}')
        return ruta

    def cargar(self, version: str) -> ConfigRAG:
        ruta = self.directorio / f'config_v{version}.json'
        if not ruta.exists():
            raise FileNotFoundError(f'No existe la versión {version}')
        with open(ruta) as f:
            datos = json.load(f)
        config = ConfigRAG(
            version=datos['version'],
            descripcion=datos['descripcion'],
            chunking=ConfigChunking(**datos['chunking']),
            retrieval=ConfigRetrieval(**datos['retrieval']),
        )
        logger.info(f'Configuración v{version} cargada')
        return config

    def listar_versiones(self) -> List[str]:
        return sorted(p.stem.replace('config_v', '') for p in self.directorio.glob('config_v*.json'))


# ── Demo: guardar dos versiones y comparar ────────────────────────────────────
gestor_config = GestorConfiguracion()

# Versión inicial
config_v1 = ConfigRAG(
    version='1.0',
    descripcion='Configuración inicial con TF-IDF básico',
    chunking=ConfigChunking(estrategia='fijo', tamano_chunk=500),
    retrieval=ConfigRetrieval(usar_reranking=False, top_k_final=10),
)

# Versión mejorada tras evaluación
config_v2 = ConfigRAG(
    version='2.0',
    descripcion='Chunking semántico + reranking activado tras mejora del 15% en MRR',
    chunking=ConfigChunking(estrategia='semantico', tamano_chunk=512),
    retrieval=ConfigRetrieval(usar_reranking=True, top_k_final=5),
)

gestor_config.guardar(config_v1)
gestor_config.guardar(config_v2)

print(f'\nVersiones disponibles: {gestor_config.listar_versiones()}')

# Simular un rollback
print('\nSimulando rollback a v1.0:')
config_activa = gestor_config.cargar('1.0')
print(f'  Descripción: {config_activa.descripcion}')
print(f'  Chunking: {config_activa.chunking.estrategia}, {config_activa.chunking.tamano_chunk} chars')
print(f'  Reranking: {config_activa.retrieval.usar_reranking}')

2026-05-18 16:57:34 | INFO     | rag                  | Configuración v1.0 guardada en /tmp/rag_configs/config_v1.0.json
2026-05-18 16:57:34 | INFO     | rag                  | Configuración v2.0 guardada en /tmp/rag_configs/config_v2.0.json
2026-05-18 16:57:34 | INFO     | rag                  | Configuración v1.0 cargada



Versiones disponibles: ['1.0', '2.0']

Simulando rollback a v1.0:
  Descripción: Configuración inicial con TF-IDF básico
  Chunking: fijo, 500 chars
  Reranking: False


## 5. Los 4 Fallos Más Comunes en RAG de Producción

Estos son los problemas que aparecen una y otra vez en sistemas RAG reales. Aprender a reconocerlos te ahorrará muchas horas de debugging.

---

### Fallo 1: El Chunking Rompe el Contexto

**Qué ocurre**: el chunk recuperado contiene la respuesta a medias — la frase importante empieza en el chunk anterior o termina en el siguiente.

**Síntomas**:
- El LLM dice "según el contexto..." pero la respuesta está incompleta
- Las respuestas mencionan conceptos que no aparecen en los chunks
- Los usuarios reportan respuestas que "se cortan" en lo importante

**Cómo diagnosticarlo**: imprime los chunks exactos que llegan al LLM y léelos.

**Solución**: añadir overlap entre chunks (ya visto en el notebook 01), o usar chunking semántico.

---

### Fallo 2: Retrieval con Alta Similitud pero Baja Relevancia

**Qué ocurre**: el sistema recupera documentos semánticamente similares a la pregunta, pero que no la responden.

**Ejemplo**: query = *"¿Cómo configuro el timeout del servidor?"* → el sistema devuelve documentos sobre "configuración general" y "servidores" que no mencionan timeouts.

**Síntomas**:
- Los scores de similitud son altos (>0.8) pero las respuestas son incorrectas
- El LLM dice que no tiene información suficiente aunque debería tenerla
- Funciona bien con preguntas genéricas pero falla con preguntas específicas

**Solución**: añadir reranking con cross-encoder (ya visto en el notebook 02). El cross-encoder entiende la intención de la pregunta mejor que la similitud coseno.

---

### Fallo 3: Latencia Excesiva en el Reranking

**Qué ocurre**: el sistema es lento porque pasa demasiados candidatos al cross-encoder.

**Síntomas**:
- El sistema tarda >2 segundos en responder
- La etapa de reranking domina el 90% del tiempo total
- La latencia aumenta con el número de usuarios simultáneos

**Causa**: el cross-encoder es O(K) — si pasas 100 candidatos en lugar de 20, tarda 5 veces más.

**Solución**: reducir `top_k_retrieval`. En la mayoría de casos, 20 candidatos para el cross-encoder es suficiente. Si necesitas más recall, optimiza la primera etapa (mejor BM25 + dense) en lugar de pasar más candidatos al reranker.

---

### Fallo 4: Deriva de Consultas (Query Drift)

**Qué ocurre**: el sistema funciona bien al lanzarlo, pero 3 meses después empieza a dar peores respuestas. Los usuarios hacen preguntas diferentes a las que usaste para configurar el sistema.

**Síntomas**:
- Las métricas de calidad empeoran gradualmente semana a semana
- Los usuarios se quejan de preguntas específicas que antes funcionaban
- Hay categorías enteras de preguntas que el sistema no sabe responder

**Solución**: guardar las queries reales de los usuarios (anonimizadas) y revisarlas cada mes. Actualizar el conjunto de evaluación con ejemplos reales, no solo los que se te ocurrieron al principio.

In [8]:
# ── Herramientas básicas de diagnóstico ───────────────────────────────────────

def diagnosticar_scores_retrieval(scores: List[float], umbral_alerta: float = 0.3) -> None:
    """
    Fallo 2: detecta si los scores son sospechosamente bajos.
    Scores bajos indican que el corpus no contiene información relevante
    para la query, o que el chunking no está funcionando bien.
    """
    score_maximo = max(scores) if scores else 0.0
    if score_maximo < umbral_alerta:
        logger.warning(
            f'Score máximo bajo ({score_maximo:.3f} < {umbral_alerta}): '
            f'el corpus puede no tener información relevante para esta query'
        )
        return False
    return True


def diagnosticar_latencia(tiempos_ms: Dict[str, float], presupuesto_ms: float = 1000.0) -> None:
    """
    Fallo 3: identifica qué etapa está consumiendo el presupuesto de latencia.
    """
    total = sum(tiempos_ms.values())
    print(f'Latencia total: {total:.0f}ms (presupuesto: {presupuesto_ms:.0f}ms)')
    for etapa, tiempo in sorted(tiempos_ms.items(), key=lambda x: x[1], reverse=True):
        porcentaje = tiempo / total * 100
        barra = '█' * int(porcentaje / 5)
        alerta = ' ← CUELLO DE BOTELLA' if porcentaje > 60 else ''
        print(f'  {etapa:<20} {tiempo:>6.0f}ms  {barra:<20} {porcentaje:.0f}%{alerta}')
    if total > presupuesto_ms:
        logger.warning(f'Presupuesto de latencia superado: {total:.0f}ms > {presupuesto_ms:.0f}ms')


def diagnosticar_query_drift(queries_recientes: List[str], queries_originales: List[str]) -> None:
    """
    Fallo 4: compara la longitud media de queries recientes vs originales
    como señal simple de drift. En producción usar embeddings para comparar.
    """
    longitud_original = sum(len(q.split()) for q in queries_originales) / len(queries_originales)
    longitud_reciente = sum(len(q.split()) for q in queries_recientes) / len(queries_recientes)
    cambio = (longitud_reciente - longitud_original) / longitud_original * 100
    print(f'Longitud media original: {longitud_original:.1f} palabras')
    print(f'Longitud media reciente: {longitud_reciente:.1f} palabras')
    print(f'Cambio: {cambio:+.1f}%')
    if abs(cambio) > 30:
        logger.warning('Posible query drift: las consultas han cambiado significativamente. Revisar el corpus.')


# ── Demos ─────────────────────────────────────────────────────────────────────
print('=== Diagnóstico de scores bajos (Fallo 2) ===')
hay_relevantes = diagnosticar_scores_retrieval([0.12, 0.09, 0.08, 0.07], umbral_alerta=0.3)
print(f'¿Se encontraron documentos relevantes? {hay_relevantes}\n')

print('=== Diagnóstico de latencia (Fallo 3) ===')
diagnosticar_latencia({
    'bm25':          8.0,
    'dense_faiss':  15.0,
    'cross_encoder': 820.0,  # el cuello de botella
    'llm':          157.0,
}, presupuesto_ms=500)

print('\n=== Diagnóstico de query drift (Fallo 4) ===')
diagnosticar_query_drift(
    queries_recientes=[
        '¿Cómo puedo configurar el timeout específico para conexiones SSL en el servidor nginx de producción?',
        '¿Cuál es la diferencia entre autenticación OAuth2 y JWT y cuándo usar cada uno en microservicios?',
        '¿Por qué mi pipeline de CI/CD falla intermitentemente solo en las horas de mayor tráfico?',
    ],
    queries_originales=[
        '¿Cómo instalar Python?',
        '¿Qué es una API?',
        '¿Cómo hacer un bucle for?',
    ]
)

2026-05-18 16:57:34 | WARNING  | rag                  | Score máximo bajo (0.120 < 0.3): el corpus puede no tener información relevante para esta query
2026-05-18 16:57:34 | WARNING  | rag                  | Presupuesto de latencia superado: 1000ms > 500ms
2026-05-18 16:57:34 | WARNING  | rag                  | Posible query drift: las consultas han cambiado significativamente. Revisar el corpus.


=== Diagnóstico de scores bajos (Fallo 2) ===
¿Se encontraron documentos relevantes? False

=== Diagnóstico de latencia (Fallo 3) ===
Latencia total: 1000ms (presupuesto: 500ms)
  cross_encoder           820ms  ████████████████     82% ← CUELLO DE BOTELLA
  llm                     157ms  ███                  16%
  dense_faiss              15ms                       2%
  bm25                      8ms                       1%

=== Diagnóstico de query drift (Fallo 4) ===
Longitud media original: 4.0 palabras
Longitud media reciente: 15.3 palabras
Cambio: +283.3%


## 6. Checklist Antes de Poner en Producción

Antes de abrir el sistema a usuarios reales, revisa estos puntos. No hace falta tenerlos todos perfectos, pero deberías saber cuáles tienes y cuáles te faltan.

### Mínimos imprescindibles
- [ ] Logging activado en todos los componentes
- [ ] Manejo de errores con fallback en retrieval y generación
- [ ] Timeout configurado en las llamadas al LLM
- [ ] Configuración en un archivo externo (no hardcodeada)
- [ ] Al menos una consulta de evaluación para detectar regresiones

### Recomendado para producción real
- [ ] Monitoreo de latencia por etapa con alertas
- [ ] Detección de scores bajos (Fallo 2)
- [ ] Actualización incremental de índices (Fallo 4)
- [ ] Plan de rollback documentado
- [ ] Logs de queries de usuarios (anonimizados) para detectar drift

In [9]:
# ── Auto-verificación básica ───────────────────────────────────────────────────
# Este script comprueba que los elementos mínimos están en orden.
# Adáptalo a tu proyecto.

def verificar_produccion(config_path: Optional[str] = None) -> None:
    checks = []

    # 1. ¿Hay logging configurado?
    tiene_logging = len(logging.getLogger().handlers) > 0
    checks.append(('Logging configurado', tiene_logging))

    # 2. ¿Existe un archivo de configuración?
    if config_path:
        tiene_config = Path(config_path).exists()
    else:
        tiene_config = any(Path('/tmp/rag_configs').glob('config_v*.json'))
    checks.append(('Configuración en archivo externo', tiene_config))

    # 3. ¿Hay al menos una versión de config guardada?
    versiones = gestor_config.listar_versiones()
    checks.append(('Al menos 1 versión de config guardada', len(versiones) > 0))

    # 4. ¿El gestor de documentos está inicializado?
    checks.append(('Gestor de documentos inicializado', len(gestor.documentos) > 0))

    # ── Mostrar resultados ─────────────────────────────────────────────────────
    pasados = sum(1 for _, ok in checks if ok)
    print(f'Verificación de producción: {pasados}/{len(checks)} checks pasados\n')
    for descripcion, ok in checks:
        icono = '✓' if ok else '✗'
        estado = 'OK' if ok else 'PENDIENTE'
        print(f'  [{icono}] {descripcion:<45} {estado}')

    if pasados < len(checks):
        print('\nAlgunos checks están pendientes. Revisa los marcados con ✗ antes del despliegue.')
    else:
        print('\nTodo OK. Listo para producción.')


verificar_produccion()

Verificación de producción: 4/4 checks pasados

  [✓] Logging configurado                           OK
  [✓] Configuración en archivo externo              OK
  [✓] Al menos 1 versión de config guardada         OK
  [✓] Gestor de documentos inicializado             OK

Todo OK. Listo para producción.


## Resumen

En este notebook hemos cubierto los aspectos prácticos para mantener un sistema RAG en producción:

| Tema | Herramienta | Para qué sirve |
|---|---|---|
| Logging | `logging` estándar de Python | Saber qué pasa cuando algo falla |
| Latencia | `Cronometro` con context manager | Identificar qué etapa es lenta |
| Errores | try/except + fallback | El sistema no se cae por un fallo puntual |
| Documentos | `GestorDocumentos` con hashes | Reindexar solo lo que cambió |
| Configuración | JSON externo + versiones | Cambiar parámetros sin tocar el código |
| Diagnóstico | Funciones de alerta | Detectar los 4 fallos comunes pronto |

### El mensaje principal

No tienes que implementarlo todo desde el primer día. El orden recomendado para un principiante:

1. **Primero**: logging y manejo de errores básico — sin esto estás ciego
2. **Después**: medir latencia por etapa — para saber dónde optimizar
3. **Cuando los datos cambien**: actualización incremental de índices
4. **Cuando tengas usuarios reales**: configuración versionada y monitoreo de drift

### Próximos pasos

- Para evaluar la calidad del sistema de forma rigurosa: ver el notebook `03_evaluation_metrics.ipynb`
- Para escalar a producción real: explorar FastAPI + Redis para caché de embeddings, y Prometheus para métricas